# 182. Self-RAG：怎样让模型按需检索、生成并自我批判？

> **面试问题：Reflection token 分别控制什么？Retrieve、ISREL、ISSUP、ISUSE 怎样组成可评测状态机？**

## 先给结论

Self-RAG 不是在普通 RAG 后加一句“请反思”，而是训练/使用显式 reflection decisions：是否需要检索、文档是否相关、生成是否被证据支持、答案是否有用。推理可用这些分数筛文档和候选，但 critic 与 generator 可能共享偏差，因此关键 claim 仍需确定性/独立验证。

## 推荐回答主线

1. 把 reflection token 解析为有限枚举，不允许自由文本悄悄改变控制流。
2. 按需触发 retrieval，对每篇 passage 评 relevance，再生成带 provenance 的 segment。
3. 对 segment 做 support 与 utility 评分，组合到 beam score，并允许拒答/重新检索。
4. 评估 retrieval rate、Recall、citation support、utility、延迟与 critic 校准，绑定 token/schema 版本。

## 教学实现边界

代码用词项重叠和受控 claim verifier 代替训练后的 reflection head，只演示状态与评分合同；真实 Self-RAG 需要相应训练数据和模型权重。

## 一手资料

- [Self-RAG](https://arxiv.org/abs/2310.11511)
- [Retrieval-Augmented Generation](https://arxiv.org/abs/2005.11401)
- [Adaptive-RAG](https://arxiv.org/abs/2403.14403)


In [ ]:
import hashlib
import json
import math
import re
from dataclasses import asdict, dataclass
from enum import Enum

import numpy as np

# 小语料带稳定 doc id 和受控事实标签，答案必须能追到具体 passage。
QUERY = "RLOO 的 baseline 如何计算？"
DOCUMENTS = [
    {"id": "d1", "text": "RLOO 对同一 prompt 的其他响应奖励取平均作为 leave-one-out baseline。", "facts": {"same_prompt", "other_rewards_mean", "leave_one_out"}},
    {"id": "d2", "text": "PPO 通常训练独立 value network 估计状态价值。", "facts": {"ppo", "value_network"}},
    {"id": "d3", "text": "巴黎是法国首都。", "facts": {"paris_capital"}},
]

assert len({doc["id"] for doc in DOCUMENTS}) == len(DOCUMENTS)
assert "baseline" in QUERY
assert all(doc["text"] and doc["facts"] for doc in DOCUMENTS)


## 1. Reflection token 是受限控制信号，不是自由 CoT

不同实现命名可变，但语义要固定：Retrieve、Relevance、Support、Utility。parser 对未知值 fail closed，防止模型输出相似字符串绕过分支。


In [ ]:
class RetrieveDecision(str, Enum):
    YES = "[Retrieve=Yes]"
    NO = "[Retrieve=No]"

class Relevance(str, Enum):
    RELEVANT = "[ISREL=Relevant]"
    IRRELEVANT = "[ISREL=Irrelevant]"

def parse_enum(raw, enum_type):
    return enum_type(raw)

# 合法 token 精确解析，未知 token 抛错，两个控制空间不可混用。
assert parse_enum("[Retrieve=Yes]", RetrieveDecision) is RetrieveDecision.YES
try:
    parse_enum("Retrieve maybe", RetrieveDecision); assert False
except ValueError:
    assert True
assert RetrieveDecision.YES.value != Relevance.RELEVANT.value


## 2. 按需检索：知识需求与模型不确定度共同决定

事实性、时效性或引用要求高时倾向检索；纯改写或已给完整上下文可跳过。模型置信必须校准，且高风险领域可由 policy 强制检索，不能完全交给模型。


In [ ]:
def need_retrieval(task_type, calibrated_confidence, requires_citation, high_risk=False):
    if not isinstance(task_type, str) or not task_type:
        raise ValueError("task_type 必须非空")
    if not isinstance(calibrated_confidence, (int, float)) or not math.isfinite(calibrated_confidence) or not 0 <= calibrated_confidence <= 1:
        raise ValueError("calibrated confidence 必须在 [0,1]")
    if type(requires_citation) is not bool or type(high_risk) is not bool:
        raise TypeError("风险与引用门禁必须是 bool")
    if high_risk or requires_citation:
        return RetrieveDecision.YES
    if task_type in {"rewrite", "summarize_given_text"}:
        return RetrieveDecision.NO
    return RetrieveDecision.YES if calibrated_confidence < 0.8 else RetrieveDecision.NO

# 引用/高风险强制检索；高置信问答和改写可走 no-retrieval；NaN 必须拒绝。
assert need_retrieval("qa", 0.95, True) is RetrieveDecision.YES
assert need_retrieval("rewrite", 0.2, False) is RetrieveDecision.NO
assert need_retrieval("qa", 0.95, False) is RetrieveDecision.NO
try:
    need_retrieval("qa", float("nan"), False); assert False
except ValueError:
    assert True


## 3. ISREL：先评 passage-query 相关，再交给 generator

检索分数与生成相关性不是同一概念。可用独立 reranker/critic 判断 passage 是否回答该 query；这里用规范化词项 Jaccard 演示，并保留原始 rank/score 供归因。


In [ ]:
def terms(text):
    return set(re.findall(r"[a-z0-9]+|[\u4e00-\u9fff]+", text.casefold()))

def relevance_score(query, passage):
    q, p = terms(query), terms(passage)
    return len(q & p) / max(len(q), 1)

def classify_relevance(score, threshold=0.2):
    if not all(isinstance(x, (int, float)) and math.isfinite(x) for x in (score, threshold)):
        raise ValueError("相关度和阈值必须有限")
    if not 0 <= score <= 1 or not 0 <= threshold <= 1:
        raise ValueError("相关度和阈值必须在 [0,1]")
    return Relevance.RELEVANT if score >= threshold else Relevance.IRRELEVANT

# RLOO 文档比巴黎文档更相关，分类枚举与阈值一致，非法阈值拒绝。
rel_scores = [relevance_score(QUERY, doc["text"]) for doc in DOCUMENTS]
assert rel_scores[0] > rel_scores[2]
assert classify_relevance(rel_scores[0]) is Relevance.RELEVANT
assert classify_relevance(rel_scores[2]) is Relevance.IRRELEVANT
try:
    classify_relevance(0.5, -0.1); assert False
except ValueError:
    assert True


## 4. ISSUP：claim 必须由指定证据蕴含，而非只主题相似

相关 passage 不一定支持最终 claim。支持检查绑定 claim、doc id、证据 span 和 verifier 版本；受控示例用关键事实集合，生产可用 NLI/规则/执行验证并抽样人审。


In [ ]:
def claim_supported(claim_facts, evidence_facts):
    claim, evidence = set(claim_facts), set(evidence_facts)
    if not claim:
        raise ValueError("claim facts 不得为空")
    return claim <= evidence

facts_by_doc = {doc["id"]: set(doc["facts"]) for doc in DOCUMENTS}

# verifier 按事实蕴含计算，不依赖 doc id；多个必要事实缺一即失败。
assert claim_supported({"other_rewards_mean", "leave_one_out"}, facts_by_doc["d1"])
assert not claim_supported({"value_network"}, facts_by_doc["d1"])
assert not claim_supported({"other_rewards_mean", "value_network"}, facts_by_doc["d1"])
try:
    claim_supported(set(), facts_by_doc["d1"]); assert False
except ValueError:
    assert True


## 5. ISUSE：有证据仍可能没有回答用户问题

utility 检查完整性、直接性、格式和安全。例如只复述相关段落但没解释公式，support 高而 utility 低。评分维度应独立记录，避免一个总分掩盖失败。


In [ ]:
def utility_score(answer, required_concepts):
    if not isinstance(answer, str):
        raise TypeError("answer 必须是字符串")
    if not required_concepts or any(not isinstance(item, str) or not item for item in required_concepts):
        raise ValueError("required concepts 必须是非空字符串列表")
    normalized = answer.casefold()
    coverage = sum(concept.casefold() in normalized for concept in required_concepts) / len(required_concepts)
    concise_bonus = 0.1 if answer and len(answer) <= 120 else 0.0
    return min(1.0, coverage + concise_bonus)

# 完整简洁答案比只说术语得分高；空约束不能制造虚假高 utility。
full_answer = "对同一 prompt，当前响应的 baseline 是其他 G-1 个响应 reward 的平均值。"
partial_answer = "RLOO 使用 baseline。"
required = ["同一 prompt", "其他", "平均"]
assert utility_score(full_answer, required) > utility_score(partial_answer, required)
assert 0 <= utility_score(full_answer, required) <= 1
assert utility_score("", required) == 0
try:
    utility_score("任意答案", []); assert False
except ValueError:
    assert True


## 6. 有界状态机：Retrieve→Filter→Generate→Critique→Accept/Retry

每一步写 trace 和预算；critique 失败可换 passage/查询或拒答，不能无限自我反思。生成结果带 doc id，避免后处理猜引用。


In [ ]:
def candidate_score(log_probability, relevance, support, utility, weights=(0.2, 0.2, 0.4, 0.2)):
    values = np.asarray([log_probability, relevance, support, utility], dtype=float)
    weight_array = np.asarray(weights, dtype=float)
    if weight_array.shape != (4,) or not np.isfinite(weight_array).all() or (weight_array < 0).any() or not math.isclose(float(weight_array.sum()), 1.0, abs_tol=1e-9):
        raise ValueError("candidate weights 必须是四个非负且和为 1 的有限数")
    if not np.isfinite(values).all() or log_probability > 0 or any(not 0 <= value <= 1 for value in (relevance, support, utility)):
        raise ValueError("candidate score 输入超出合法范围")
    return float(values @ weight_array)

def generate_candidate(document):
    evidence_facts = set(document.get("facts", ()))
    if {"other_rewards_mean", "leave_one_out"} <= evidence_facts:
        return {"answer": full_answer, "facts": {"other_rewards_mean", "leave_one_out"}}
    return {"answer": partial_answer, "facts": {"baseline"}}

def make_retriever(documents, counter):
    def retrieve(query):
        counter["calls"] += 1
        return list(documents)
    return retrieve

def run_self_rag(query, retriever, task_type="qa", calibrated_confidence=0.0, requires_citation=True, high_risk=False, max_retries=2, no_retrieval_answer="", score_weights=(0.2, 0.2, 0.4, 0.2)):
    if type(max_retries) is not int or max_retries < 0:
        raise ValueError("max_retries 必须是非负整数")
    if not isinstance(query, str) or not query or not callable(retriever):
        raise ValueError("query 与 retriever 非法")
    # 配置在分支前统一校验，No-Retrieval 也不能绕过非法 score weights。
    candidate_score(0.0, 0.0, 0.0, 0.0, score_weights)
    decision = need_retrieval(task_type, calibrated_confidence, requires_citation, high_risk)
    trace = [{"state": "decide", "decision": decision.value}]
    if decision is RetrieveDecision.NO:
        useful = utility_score(no_retrieval_answer, required) >= 0.8
        trace.append({"state": "critique", "supported": None, "useful": useful, "mode": "no_retrieval"})
        if useful:
            return {"status": "accepted_no_retrieval", "answer": no_retrieval_answer, "citation": None, "trace": trace}
        return {"status": "abstain", "trace": trace}

    # 只有 Retrieve=Yes 后才调用 retriever；no-retrieval 分支是真正的零召回。
    documents = list(retriever(query))
    ids = [doc.get("id") for doc in documents]
    if any(not doc_id for doc_id in ids) or len(ids) != len(set(ids)):
        raise ValueError("document id 必须非空且唯一")
    ranked = sorted(documents, key=lambda doc: relevance_score(query, doc["text"]), reverse=True)
    accepted_candidates = []
    for doc in ranked[:max_retries]:
        relevance = relevance_score(query, doc["text"])
        rel = classify_relevance(relevance)
        trace.append({"state": "filter", "doc": doc["id"], "relevance": rel.value})
        if rel is Relevance.IRRELEVANT:
            continue
        candidate = generate_candidate(doc)
        supported = claim_supported(candidate["facts"], doc.get("facts", ()))
        utility = utility_score(candidate["answer"], required)
        score = candidate_score(doc.get("log_probability", -0.5), relevance, float(supported), utility, score_weights)
        trace.append({"state": "critique", "doc": doc["id"], "supported": supported, "utility": utility, "score": score})
        if supported and utility >= 0.8:
            accepted_candidates.append((score, doc["id"], candidate["answer"]))
    if accepted_candidates:
        score, doc_id, answer = max(accepted_candidates, key=lambda item: (item[0], item[1]))
        trace.append({"state": "select", "doc": doc_id, "score": score})
        return {"status": "accepted", "answer": answer, "citation": doc_id, "trace": trace}
    return {"status": "abstain", "trace": trace}

# 主路径覆盖延迟召回、真正零召回、事实 verifier、四信号选优和负预算拒绝。
retrieval_calls = {"calls": 0}
self_rag_result = run_self_rag(QUERY, make_retriever(DOCUMENTS, retrieval_calls), calibrated_confidence=0.4, requires_citation=True)
assert self_rag_result["status"] == "accepted" and self_rag_result["citation"] == "d1"
assert retrieval_calls["calls"] == 1 and any(event["state"] == "select" for event in self_rag_result["trace"])

def forbidden_retriever(query):
    raise AssertionError("no-retrieval 分支不得召回")

no_retrieval_result = run_self_rag(QUERY, forbidden_retriever, calibrated_confidence=0.95, requires_citation=False, max_retries=0, no_retrieval_answer=full_answer)
assert no_retrieval_result["status"] == "accepted_no_retrieval"
assert not any(event["state"] == "filter" for event in no_retrieval_result["trace"])
unsupported_docs = [{**doc, "facts": set()} for doc in DOCUMENTS]
assert run_self_rag(QUERY, make_retriever(unsupported_docs, {"calls": 0}), calibrated_confidence=0.2, max_retries=2)["status"] == "abstain"
ranked_candidates = [
    {**DOCUMENTS[0], "id": "low", "log_probability": -2.0},
    {**DOCUMENTS[0], "id": "high", "log_probability": -0.1},
]
selected = run_self_rag(QUERY, make_retriever(ranked_candidates, {"calls": 0}), calibrated_confidence=0.2, max_retries=2)
assert selected["citation"] == "high"
try:
    run_self_rag(QUERY, forbidden_retriever, max_retries=-1); assert False
except ValueError:
    assert True


## 7. 候选评分：生成似然、relevance、support、utility 分开加权

Self-RAG 可用 reflection 分数控制 beam。权重反映应用目标，不能在测试集调到最好后当通用配置；关键事实场景通常对 unsupported 设硬门禁，而不是让高流畅度抵消。


In [ ]:
# candidate score 已被主 loop 消费；unsupported 候选不能仅靠流畅度覆盖，非法权重 fail closed。
grounded = candidate_score(-0.5, 0.9, 1.0, 0.9)
fluent_unsupported = candidate_score(-0.1, 0.9, 0.0, 0.9)
assert grounded > fluent_unsupported
assert math.isfinite(grounded)
assert any(event["state"] == "critique" and "score" in event for event in self_rag_result["trace"])
try:
    candidate_score(-0.1, 0.9, 0.0, 0.9, (0.5, 0.2, -0.1, 0.4)); assert False
except ValueError:
    assert True
try:
    run_self_rag(QUERY, forbidden_retriever, calibrated_confidence=0.95, requires_citation=False, no_retrieval_answer=full_answer, score_weights=(1.0, 0.0, 0.0)); assert False
except ValueError:
    assert True


## 8. 评测与版本：critic 也要校准和防污染

报告 retrieval decision precision/recall、passage relevance、claim support、citation precision、utility、abstain、重试与延迟。generator/critic 同源可能共错，应加规则/独立模型/人审，并绑定 reflection token ids。


In [ ]:
@dataclass(frozen=True)
class SelfRAGArtifact:
    generator: str
    retriever_index: str
    critic: str
    reflection_schema: str
    thresholds: tuple[float, ...]
    max_retries: int

def artifact_hash(artifact):
    return hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()

# 阈值与重试预算进入摘要，核心组件身份非空。
artifact = SelfRAGArtifact("gen-v4", "index-v8", "critic-v3", "reflection-v2", (0.2, 0.8), 2)
digest = artifact_hash(artifact)
assert all((artifact.generator, artifact.retriever_index, artifact.critic))
assert len(digest) == 64
assert digest != artifact_hash(SelfRAGArtifact("gen-v4", "index-v8", "critic-v3", "reflection-v2", (0.3, 0.8), 2))


## 面试收束：Agent/RAG 的算法只是控制面的一部分

推荐回答顺序是：任务目标和失败代价、状态/事件/证据合同、决策公式、可执行反例、离线与在线指标、权限和版本。受控环境只能证明状态机和数值关系，不能冒充开放网络、真实用户或真实模型结果。生产系统还要处理并发、超时、幂等、恶意内容、隐私、审计、灰度与回滚。

遇到追问时，主动区分模型判断与确定 verifier、计划与真实副作用、原始 observation 与 belief/memory、召回质量与生成归因，以及多尝试成功率与单次可靠性。
